# FAZ 2 — Öznitelik Analizi
Form, saha formu ve H2H özniteliklerinin keşifsel analizi.

> Kernel olarak proje `.venv`'ini seç.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from config.settings import DatabaseSettings, LEAGUES
from db.connection import DatabaseManager
from models.dataset import MatchDataLoader
from models.features import FeatureBuilder

db = DatabaseManager(DatabaseSettings.from_env())
matches = MatchDataLoader(db.engine).load()
fs = FeatureBuilder().build(matches)
print(len(fs.train), 'eğitim maçı |', len(fs.upcoming), 'yaklaşan maç |', len(fs.feature_columns), 'öznitelik')

## Hedef dağılımı (H / D / A) — lig bazında

In [ ]:
(fs.train.assign(lig=fs.train['lig_id'].map(LEAGUES))
   .groupby('lig')['sonuc'].value_counts(normalize=True).unstack().round(3))

## Eksik değer oranları
xG eski sezonlarda yoktur; model bunu NaN olarak görecek (XGBoost NaN'ı doğal olarak işler).

In [ ]:
fs.train[fs.feature_columns].isna().mean().sort_values(ascending=False).head(15).round(3)

## Öznitelik ↔ sonuç ilişkisi
Fark öznitelikleri ile ev sahibi galibiyeti arasındaki korelasyon.

In [ ]:
t = fs.train.assign(ev_kazandi=(fs.train['sonuc'] == 'H').astype(int))
fark_cols = [c for c in fs.feature_columns if c.startswith(('fark_', 'h2h_'))]
t[fark_cols + ['ev_kazandi']].corr()['ev_kazandi'].drop('ev_kazandi').sort_values()

## Puan formu farkına göre sonuç dağılımı

In [ ]:
t['fark_bin'] = pd.qcut(t['fark_genel_puan_ort'], 5)
t.groupby('fark_bin', observed=True)['sonuc'].value_counts(normalize=True).unstack().round(3)